In [ ]:
# Basic imports
import io
import json
import uuid
from pprint import pprint
from tqdm import tqdm

# Third party core
import boto3
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Sagemaker imports
import sagemaker
from sagemaker import ModelPackage
from sagemaker.deserializers import JSONDeserializer
from sagemaker.model_monitor import (
    DataCaptureConfig,
    DatasetFormat,
    DefaultModelMonitor,
    ModelQualityMonitor,
)
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer

In [ ]:
%store -r

In [ ]:
%store

In [ ]:
bucket = bucket
project_prefix = project_prefix

s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)

# Deploying Registered Models
Both endpoints serve the model versions that passed their pipeline's quality gate and were registered by `5_Classification_Model` and `5-1_Keypoint_Regression_Model`. Data capture records every request and response.

In [ ]:
# Endpoints
endpoint_id = uuid.uuid4()
kp_endpoint_name = f"keypoint-regression-{endpoint_id}"
cls_endpoint_name = f"cls-{endpoint_id}"

# Registered model versions
kp_model = ModelPackage(
    role=role,
    model_package_arn=kp_model_package_arn,
    sagemaker_session=sagemaker_session,
)

cls_model = ModelPackage(
    role=role,
    model_package_arn=cls_model_package_arn,
    sagemaker_session=sagemaker_session,
)

# S3 locations for captured traffic and baselines
monitoring_prefix = f"{project_prefix}/monitoring/{endpoint_id}"
data_capture_prefix = f"{monitoring_prefix}/datacapture"
baseline_prefix = f"{monitoring_prefix}/baselines"

In [ ]:
# Deploying Keypoint model
kp_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=kp_endpoint_name,
    data_capture_config=DataCaptureConfig(
        enable_capture=True, 
        sampling_percentage=100, 
        destination_s3_uri=f"s3://{bucket}/{data_capture_prefix}/keypoint_regressor"
    ),
)

In [ ]:
# Deploying Classification model
cls_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=cls_endpoint_name,
    data_capture_config=DataCaptureConfig(
        enable_capture=True, 
        sampling_percentage=100, 
        destination_s3_uri=f"s3://{bucket}/{data_capture_prefix}/classifier"
    ),
)

In [ ]:
# Both endpoints receive raw image bytes, the way a production client would send them
kp_predictor = Predictor(
    endpoint_name=kp_endpoint_name, 
    sagemaker_session=sagemaker_session,
    serializer=IdentitySerializer("image/jpeg"),
    deserializer=JSONDeserializer(),
)

cls_predictor = Predictor(
    endpoint_name=cls_endpoint_name, 
    sagemaker_session=sagemaker_session,
    serializer=IdentitySerializer("image/jpeg"),
    deserializer=JSONDeserializer(),
)

%store kp_endpoint_name cls_endpoint_name

# Baseline Traffic
Each model's held-out test split is sent through its endpoint. Every request carries a unique inference ID, which is logged with the image and its ground truth. Model Monitor uses these IDs to join captured predictions with ground-truth labels.

Baselines are computed from the endpoint's responses rather than from offline predictions, so they reflect the serving path that will be monitored.

In [ ]:
def read_s3_bytes(s3_uri: str) -> bytes:
    bucket_name, key = s3_uri.replace("s3://", "").split("/", 1)
    return s3.get_object(Bucket=bucket_name, Key=key)["Body"].read()


def invoke(predictor, s3_uri: str):
    """Send raw image bytes to an endpoint, tagged with a unique inference ID."""
    inference_id = str(uuid.uuid4())
    response = predictor.predict(read_s3_bytes(s3_uri), inference_id=inference_id)
    return inference_id, response


def upload_csv(df: pd.DataFrame, key: str) -> str:
    s3.put_object(Bucket=bucket, Key=key, Body=df.to_csv(index=False).encode())
    return f"s3://{bucket}/{key}"


def registered_evaluation(model_package_arn: str) -> dict:
    """Evaluation report the training pipeline attached to a registered model."""
    package = sagemaker_client.describe_model_package(ModelPackageName=model_package_arn)
    return json.loads(read_s3_bytes(package["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]))


def summarize_feature_statistics(baseline_job) -> pd.DataFrame:
    """One row per feature from a data-quality baseline's statistics.json."""
    rows = []
    for feature in baseline_job.baseline_statistics().body_dict["features"]:
        stats = feature.get("numerical_statistics", {})
        rows.append({
            "feature": feature["name"],
            "type": feature["inferred_type"],
            "mean": stats.get("mean"),
            "std_dev": stats.get("std_dev"),
            "min": stats.get("min"),
            "max": stats.get("max"),
        })
    return pd.DataFrame(rows).set_index("feature")

## Classifier

In [ ]:
# Test split of the training manifest used by 5_Classification_Model
training_manifest = pd.read_parquet(io.BytesIO(read_s3_bytes(f"s3://{bucket}/{training_manifest_remote_path}")))
cls_test = training_manifest[training_manifest["split"] == "test"]

rows = []
for uri, label in tqdm(zip(cls_test["remote_path"], cls_test["label"]), total=len(cls_test),
                       desc="Classifier baseline traffic", unit="image"):
    inference_id, result = invoke(cls_predictor, uri)
    rows.append({
        "inference_id": inference_id,
        "remote_path": uri,
        "label": int(label),
        "prediction": int(result["predicted_class"]),
        "probability": float(result["probabilities"]["cat"]),
    })

cls_baseline_df = pd.DataFrame(rows)
cls_baseline_df["label"].value_counts()

In [ ]:
# Train/serve consistency: the pipeline's offline evaluation and the deployed endpoint
# score the same test split. A large gap would point to preprocessing skew.
offline = registered_evaluation(cls_model_package_arn)["binary_classification_metrics"]

endpoint_metrics = {
    "accuracy": accuracy_score(cls_baseline_df["label"], cls_baseline_df["prediction"]),
    "precision": precision_score(cls_baseline_df["label"], cls_baseline_df["prediction"]),
    "recall": recall_score(cls_baseline_df["label"], cls_baseline_df["prediction"]),
    "f1": f1_score(cls_baseline_df["label"], cls_baseline_df["prediction"]),
    "auc": roc_auc_score(cls_baseline_df["label"], cls_baseline_df["probability"]),
}

display(pd.DataFrame({
    "pipeline_evaluation": {name: offline[name]["value"] for name in endpoint_metrics},
    "endpoint": endpoint_metrics,
}).round(4))

print(classification_report(cls_baseline_df["label"], cls_baseline_df["prediction"],
                            target_names=["non_cat", "cat"], digits=4))

In [ ]:
cls_baseline_prefix = f"{baseline_prefix}/classifier"

# Model-quality baseline dataset: ground truth, predicted label, cat probability
cls_quality_dataset_uri = upload_csv(
    cls_baseline_df[["label", "prediction", "probability"]],
    f"{cls_baseline_prefix}/model-quality-dataset.csv",
)

# Prediction-drift baseline dataset: endpoint outputs only, no labels needed
cls_prediction_dataset_uri = upload_csv(
    cls_baseline_df[["prediction", "probability"]],
    f"{cls_baseline_prefix}/prediction-dataset.csv",
)

# Inference log: which inference ID was sent with which image and label
upload_csv(cls_baseline_df, f"{cls_baseline_prefix}/inference-log.csv")

### Model-quality baseline
Metrics and suggested constraints from predictions against ground truth. Model Monitor compares live traffic to these once ground-truth labels arrive.

In [ ]:
cls_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

cls_quality_monitor.suggest_baseline(
    job_name=f"cls-quality-baseline-{endpoint_id}",
    baseline_dataset=cls_quality_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="BinaryClassification",
    inference_attribute="prediction",
    probability_attribute="probability",
    ground_truth_attribute="label",
    output_s3_uri=f"s3://{bucket}/{cls_baseline_prefix}/model-quality",
    wait=True,
    logs=False,
)

In [ ]:
cls_quality_baseline_job = cls_quality_monitor.latest_baselining_job

cls_quality_statistics = cls_quality_baseline_job.baseline_statistics().body_dict["binary_classification_metrics"]
display(pd.DataFrame(
    {name: metric for name, metric in cls_quality_statistics.items() if isinstance(metric, dict) and "value" in metric}
).T)

pd.DataFrame(cls_quality_baseline_job.suggested_constraints().body_dict["binary_classification_constraints"]).T

### Prediction-drift baseline
Distribution of the endpoint's outputs (predicted class and cat probability). Drift in these needs no labels, so it can be detected as soon as traffic arrives.

In [ ]:
cls_prediction_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

cls_prediction_monitor.suggest_baseline(
    job_name=f"cls-prediction-baseline-{endpoint_id}",
    baseline_dataset=cls_prediction_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{cls_baseline_prefix}/prediction-drift",
    wait=True,
    logs=False,
)

In [ ]:
cls_prediction_baseline_job = cls_prediction_monitor.latest_baselining_job

display(summarize_feature_statistics(cls_prediction_baseline_job))
pprint(cls_prediction_baseline_job.suggested_constraints().body_dict["monitoring_config"])

## Keypoint Regressor

In [ ]:
KP_NAMES = [
    "left_eye", "right_eye", "mouth",
    "left_ear_1", "left_ear_2", "left_ear_3",
    "right_ear_1", "right_ear_2", "right_ear_3",
]

KP_COLS_NORM = [f"{kp}_{axis}_norm" for kp in KP_NAMES for axis in ("x", "y")]
KP_COLS_PRED = [f"{kp}_{axis}_pred" for kp in KP_NAMES for axis in ("x", "y")]


def keypoint_nme(pred_norm, gt_norm, width, height) -> np.ndarray:
    """Per-keypoint pixel error divided by the ground-truth inter-ocular distance.
    Same definition as src/keypoint_regression/evaluation.py."""
    scale = np.array([width, height], dtype=float)
    pred_px = np.asarray(pred_norm, dtype=float).reshape(-1, 2) * scale
    gt_px = np.asarray(gt_norm, dtype=float).reshape(-1, 2) * scale
    inter_ocular = np.linalg.norm(gt_px[0] - gt_px[1])
    if inter_ocular == 0:
        # Coinciding eye annotations give no usable scale; excluded, as in evaluation.py
        return np.full(len(KP_NAMES), np.nan)
    return np.linalg.norm(pred_px - gt_px, axis=1) / inter_ocular

In [ ]:
# Test split processed by 5-1_Keypoint_Regression_Model: normalized ground truth,
# original image size, and the source image in S3
kp_test = pd.read_parquet(io.BytesIO(read_s3_bytes(f"s3://{bucket}/{kp_test_data_prefix}/annotations.parquet")))

rows = []
for _, row in tqdm(kp_test.iterrows(), total=len(kp_test), desc="Keypoint baseline traffic", unit="image"):
    inference_id, result = invoke(kp_predictor, row["remote_path"])
    pred = [result[name][axis] for name in KP_NAMES for axis in ("x", "y")]
    nme = keypoint_nme(pred, row[KP_COLS_NORM].to_numpy(), row["width"], row["height"])
    rows.append({
        "inference_id": inference_id,
        "remote_path": row["remote_path"],
        **dict(zip(KP_COLS_PRED, pred)),
        # Predicted eye distance in normalized coordinates: a label-free proxy for face size
        "pred_inter_ocular": float(np.hypot(pred[0] - pred[2], pred[1] - pred[3])),
        "nme": float(np.mean(nme)),
        **{f"nme_{name}": float(value) for name, value in zip(KP_NAMES, nme)},
    })

kp_baseline_df = pd.DataFrame(rows)
kp_baseline_df["nme"].describe()

In [ ]:
# Train/serve consistency: pipeline evaluation vs. the deployed endpoint on the same test split.
# The pipeline scores preprocessed images; the endpoint resizes the original JPEG itself.
offline = registered_evaluation(kp_model_package_arn)
valid_nme = kp_baseline_df["nme"].dropna()

kp_endpoint_metrics = {
    "nme_mean": valid_nme.mean(),
    "nme_median": valid_nme.median(),
    "nme_p95": valid_nme.quantile(0.95),
    "failure_rate_at_0_1": (valid_nme > 0.1).mean(),
}

display(pd.DataFrame({
    "pipeline_evaluation": {name: offline["regression_metrics"][name]["value"] for name in kp_endpoint_metrics},
    "endpoint": kp_endpoint_metrics,
}).round(4))

pd.DataFrame({
    "pipeline_evaluation": offline["per_keypoint_nme"],
    "endpoint": {name: kp_baseline_df[f"nme_{name}"].mean() for name in KP_NAMES},
}).round(4)

### Model-quality baseline
Model Monitor's model-quality jobs compare one predicted value with one label, so they cannot score 9 keypoints at once. The keypoint quality baseline uses NME instead. It is written in Model Monitor's statistics/constraints layout, so a custom monitoring job can check live NME against it once ground truth arrives. As with Model Monitor's suggested constraints, each baseline value is its own threshold.

In [ ]:
kp_baseline_prefix = f"{baseline_prefix}/keypoint"

kp_quality_statistics = {
    "version": 0.0,
    "dataset": {
        "item_count": int(valid_nme.size),
        "excluded_zero_inter_ocular": int(kp_baseline_df["nme"].isna().sum()),
    },
    "keypoint_metrics": {name: {"value": float(value)} for name, value in kp_endpoint_metrics.items()},
    "per_keypoint_nme": {name: {"value": float(kp_baseline_df[f"nme_{name}"].mean())} for name in KP_NAMES},
}

kp_quality_constraints = {
    "version": 0.0,
    "keypoint_constraints": {
        name: {"threshold": float(value), "comparison_operator": "GreaterThanThreshold"}
        for name, value in kp_endpoint_metrics.items()
    },
}

for name, body in [("statistics.json", kp_quality_statistics), ("constraints.json", kp_quality_constraints)]:
    s3.put_object(
        Bucket=bucket,
        Key=f"{kp_baseline_prefix}/model-quality/{name}",
        Body=json.dumps(body, indent=2).encode(),
    )

pd.DataFrame(kp_quality_constraints["keypoint_constraints"]).T

In [ ]:
# Prediction-drift baseline dataset: predicted coordinates and eye distance, no labels needed
kp_prediction_dataset_uri = upload_csv(
    kp_baseline_df[KP_COLS_PRED + ["pred_inter_ocular"]],
    f"{kp_baseline_prefix}/prediction-dataset.csv",
)

# Inference log: which inference ID was sent with which image, plus its per-image error
upload_csv(kp_baseline_df, f"{kp_baseline_prefix}/inference-log.csv")

### Prediction-drift baseline
Distribution of predicted keypoint coordinates and predicted eye distance. A shift in these flags changes in framing or face size without waiting for labels.

In [ ]:
kp_prediction_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

kp_prediction_monitor.suggest_baseline(
    job_name=f"kp-prediction-baseline-{endpoint_id}",
    baseline_dataset=kp_prediction_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{kp_baseline_prefix}/prediction-drift",
    wait=True,
    logs=False,
)

In [ ]:
kp_prediction_baseline_job = kp_prediction_monitor.latest_baselining_job

summarize_feature_statistics(kp_prediction_baseline_job)

In [ ]:
# Baseline locations for the monitoring schedules
%store monitoring_prefix data_capture_prefix baseline_prefix

# Monitoring Schedules
Scheduled monitors that use these baselines are set up in the next stage of the project.

# Deleting endpoint

In [ ]:
""" sagemaker_client.delete_endpoint(EndpointName=kp_endpoint_name)
sagemaker_client.delete_endpoint(EndpointName=cls_endpoint_name) """